<div dir="rtl">

# دسته‌بندی گزارش‌های حوادث دریایی MAIB: اجرای کامل پروژه

این نوت‌بوک گام‌های «روش تحقیق» پروپوزال را به ترتیب اجرا می‌کند. هر سلول فقط یک فایل `stepN_*.py` را اجرا می‌کند.
کد اصلی داخل همان فایل‌هاست و هر فایل در ابتدایش توضیح می‌دهد چه کاری انجام می‌دهد.

| سلول | فایل | گام پروپوزال |
|---|---|---|
| ۱ | `step1_prepare_data.py` | گام ۱: دادگان، پاک‌سازی، تقسیم ۷۰/۱۵/۱۵ |
| ۲ | `step2_tfidf_svm.py` | گام ۲-الف: خط مبنای TF-IDF + SVM (با و بدون oversampling) |
| ۳ | `step4_run_experiments.py` (که `step3_train.py` را صدا می‌زند) | گام‌های ۲-ب، ۲-ج، ۳ و ۴ |
| ۴ | `step4_run_experiments.py --oversample` | آزمایش تکمیلی oversampling |
| ۵ | `step5_report.py` | گام ۵: جدول‌ها و ماتریس درهم‌ریختگی |
| ۶ | `step6_attention.py` | گام ۵: تحلیل کیفی وزن‌های توجه |

**زمان روی GTX 1650 Ti:** هر اجرای BERT حدود ۷۵ دقیقه؛ ۱۲ اجرای اصلی حدود ۱۵ ساعت و ۲ اجرای oversampling حدود ۲٫۵ ساعت.
اجراهایی که فایل نتیجه‌شان در `results/` هست رد می‌شوند، پس اگر اجرا قطع شد، همان سلول را دوباره اجرا کن.

</div>

<div dir="rtl">

## ۰. آماده‌سازی

</div>

In [ ]:
from IPython.display import HTML, Image, display

import torch

SEEDS = "1 2"          # the seeds of the reported results; use "1 2 3" for a third seed
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - training will be very slow")

<div dir="rtl">

## ۱. دادگان (گام ۱ پروپوزال)
بارگذاری ۵٬۷۶۸ گزارش، پاک‌سازی، حذف تکراری‌ها، حذف کلاس‌های کمتر از ۲۰ گزارش و تقسیم stratified.

</div>

In [ ]:
!python step1_prepare_data.py
display(Image("reports/class_distribution.png"))

<div dir="rtl">

## ۲. خط مبنای کلاسیک: TF-IDF + SVM (گام ۲-الف)
چند ثانیه روی CPU. عددی که BERT باید از آن بهتر شود.

</div>

In [ ]:
!python step2_tfidf_svm.py

<div dir="rtl">

## ۳. آموزش مدل‌های BERT (گام‌های ۲-ب، ۲-ج، ۳ و ۴)
۶ پیکربندی × ۲ seed = **۱۲ اجرا**. فهرست پیکربندی‌ها در `config.py` و متغیر `EXPERIMENTS` است.

</div>

In [ ]:
!python step4_run_experiments.py --seeds $SEEDS

<div dir="rtl">

## ۴. آزمایش تکمیلی: oversampling (درخواست استاد راهنما)
مدل پیشنهادی با `ce`، وقتی کلاس‌های کمیاب در آموزش بیشتر دیده می‌شوند. ۲ اجرا.

</div>

In [ ]:
!python step4_run_experiments.py --oversample --seeds $SEEDS

<div dir="rtl">

## ۵. نتایج (گام ۵)
جدول اصلی (میانگین ± انحراف معیار روی seedها)، جواب دو سؤال پژوهش و oversampling، recall هر کلاس و ماتریس درهم‌ریختگی.

</div>

In [ ]:
!python step5_report.py
display(Image("reports/per_class_recall.png"))
display(Image("reports/confusion_matrix.png"))

<div dir="rtl">

## ۶. تحلیل وزن‌های توجه (گام ۵، تحلیل کیفی)
برای هر کلاس، نمونه‌های درست و نادرست با رنگ‌آمیزی کلمات بر اساس وزن توجه.

</div>

In [ ]:
!python step6_attention.py
display(HTML(open("reports/attention_examples.html").read()))

<div dir="rtl">

## ۷. افزایش داده با مدل زبانی (درخواست استاد راهنما)
گزارش‌های آموزش کلاس‌های کمیاب با یک مدل زبانی کوچک بازنویسی می‌شوند تا هر کلاس ۴۰۰ گزارش آموزش داشته باشد. val و test دست نمی‌خورند.
قبل از آموزش، نمونه‌های `reports/augmentation_examples.md` را با چشم بررسی کن.

</div>

In [ ]:
!python step7_augment.py
!python step2_tfidf_svm.py
!python step4_run_experiments.py --augment --seeds $SEEDS

<div dir="rtl">

## ۸. BiLSTM بزرگ‌تر (پرسش استاد راهنما)
مدل پیشنهادی با wce و BiLSTM بزرگ‌تر: ۳۸۴ واحد (هم‌اندازه‌ی خروجی BERT) و/یا ۲ لایه. اندازه‌ها در `config.BILSTM_SIZES` است.

</div>

In [ ]:
!python step4_run_experiments.py --bilstm-sizes --seeds $SEEDS
!python step5_report.py